# 枚举与常量

学习目标：能理解枚举成员类型与实际对象，比较 const enum、字面量联合和常量对象的适用边界。

前置知识：TypeScript 字面量联合、类型和值位置、as const 与穷尽检查；JavaScript 对象属性与位运算。

适用版本：TypeScript 7.0.2、Node.js 24.11.0；ES 模块，开启 strict。本章附加选项：isolatedModules=false、preserveConstEnums=false，含义见对应知识点。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/typescript。

配套脚本：位于 scripts/11-enums-and-constants/。

1. [main.ts](scripts/11-enums-and-constants/main.ts)：按正文顺序组织的正常示例，片段依赖同文件前文定义。
2. [type-errors.ts](scripts/11-enums-and-constants/type-errors.ts)：与正常示例隔离的类型反例，不生成或执行 JavaScript。
3. [tsconfig.json](scripts/11-enums-and-constants/tsconfig.json)、[tsconfig.errors.json](scripts/11-enums-and-constants/tsconfig.errors.json)：分别明确正常与反例文件范围。
4. [tsconfig.preserve.json](scripts/11-enums-and-constants/tsconfig.preserve.json)：单独说明和执行的配套边界示例。



Step 1：检查正常项目的类型。

```bash
npm run check:11
```

Step 2：生成正常项目的 JavaScript。

```bash
npm run build:11
```

Step 3：运行正常示例。

```bash
npm run run:11
```

Step 4：检查下文独立列出的类型反例。

```bash
npm run errors:11
# 预期非零退出；按反例注释逐行核对具体错误，不运行 type-errors.ts。
```

正常配置只包含上面列出的正常与独立运行示例，生成文件位于 .build/11-enums-and-constants/。错误配置继承正常选项，改用 type-errors.ts 并开启 noEmit。

## 1 数值枚举与命名常量

enum 为一组值提供名称，并通常生成运行时对象。数值枚举首个未初始化成员从 0 开始；后续未初始化成员可在前一个常量数值上递增。不要在需要稳定协议值时依赖随成员插入而变化的自动编号。

数值成员同时生成从名称到数值、从数值到名称的映射。反向映射是普通对象属性写入；若多个成员值相同，后写的名称会覆盖同一个反向键，因此不能把它当作一对一映射。

```typescript
export {};
enum Phase { Queued, Running = 3, Done }
enum Alias { First = 1, Second = 1 }
const phase: Phase = Phase.Done;
console.log(Phase.Queued, phase, Phase[3], Alias[1]); // 0 4 Running Second
```

以下片段来自独立的 type-errors.ts：

```typescript
enum KnownPhase { Queued, Done }
const unrelatedLiteral: KnownPhase = 99; // 该数值字面量不是任何成员。
```

## 2 字符串枚举与成员类型

字符串枚举成员需显式初始化为字符串字面量或其他字符串枚举成员，没有自动递增，也不会生成数值枚举那样的反向映射。它可以使日志和持久化值更易读，但协议兼容仍由设计者负责。

成员可在类型位置表示该成员，整个枚举可作为成员的联合参与收窄。普通字符串即使文本相同，也不能直接代替字符串枚举成员类型。这里借助 kind: Status.Ready 描述只允许就绪的结构。

```typescript
enum Status { Ready = "ready", Failed = "failed" }
type ReadyItem = { kind: Status.Ready; count: number };
const ready: ReadyItem = { kind: Status.Ready, count: 2 };
function statusLabel(value: Status): string {
  switch (value) {
    case Status.Ready: return "就绪";
    case Status.Failed: return "失败";
    default: { const rest: never = value; return rest; }
  }
}
console.log(statusLabel(ready.kind), Status.Failed, "ready" in Status); // 就绪 failed false
```

以下片段来自独立的 type-errors.ts：

```typescript
enum State { Ready = "ready", Failed = "failed" }
const textIsNotMember: State = "ready"; // 普通字符串字面量不是该字符串枚举成员。
const wrongMember: State.Ready = State.Failed; // 成员类型不同。
```

## 3 常量成员与计算成员

枚举的常量表达式有受限制的语法，包含枚举常量引用、字面量、括号和允许的算术或位运算。普通表达式能运行得出数值，不代表它就是编译期枚举常量表达式。

下面 FromText 的字符串属性读取属于计算成员；计算成员之后需要显式提供下个初始值，不能自动推导递增。TypeScript 5.0 起也为计算成员建立独立成员类型，使其可以参与联合收窄；这不表示编译器能求出任意函数调用的运行时结果。

```typescript
enum Size {
  Fixed = 1 << 1,
  Combined = Fixed | 1,
  FromText = "abc".length,
  Explicit = 9,
}
const computedMember: Size.FromText = Size.FromText;
console.log(Size.Fixed, Size.Combined, computedMember, Size.Explicit); // 2 3 3 9
```

以下片段来自独立的 type-errors.ts：

```typescript
enum MissingInitializer { Computed = "abc".length, Next } // 计算成员之后必须显式初始化。
const enum NotConstant { Length = "abc".length } // const enum 只能使用允许的常量表达式。
```

## 4 const enum 的擦除与内联

本章正常项目显式设置 isolatedModules=false、preserveConstEnums=false，使用 tsc 项目编译观察 const enum 内联。const enum 只能含常量枚举表达式；其声明被移除，成员使用位置替换为值。不能把整个 const enum 当作普通运行时对象枚举其键。

下面 Access.All 被替换为数值 3；普通 Phase、Status、Size 仍在输出中创建对象。构建后查看 .build/11-enums-and-constants/main.js，可以把类型语法擦除与需要生成的枚举代码分开识别。

```typescript
const enum Access { Read = 1, Write = 2, All = Read | Write }
const selected = Access.All;
console.log(selected, (selected & Access.Read) !== 0); // 3 true
```

以下片段来自独立的 type-errors.ts：

```typescript
const enum Inlined { First = 1 }
const enumObject = Inlined; // 不能把 const enum 本身作为一般运行时值。
```

## 5 保留声明与跨包限制

preserveConstEnums=true 可以保留 const enum 的运行时对象声明；在本章项目编译对照中，使用位置仍可内联。配套 tsconfig.preserve.json 只改变这个选项和输出目录，用相同源码比较，避免两次输出互相覆盖。

跨包公开的 ambient const enum（环境常量枚举）常见于 .d.ts 声明文件。单文件转换工具不能依赖跨文件类型信息来替换其值，因此 isolatedModules 下禁止访问这种成员。另一个风险是编译时把依赖版本 A 的值内联，运行时却载入版本 B，造成值不一致。

声明文件本身不提供 JavaScript 实现。若要发布稳定的枚举 API，可用普通 enum 或常量对象；也可在经过设计的构建中保留对象并从发布声明移除 const，但只开 preserveConstEnums 并不会自动完成这一步。下面反例只做类型检查，模拟环境声明边界，不伪造已安装的包。

Step 1：生成保留 const enum 声明的对照输出。

```bash
npm run build:11:preserve
```

Step 2：运行对照输出，结果应与 run:11 一致。

```bash
npm run run:11:preserve
```

类型反例配置额外设置 isolatedModules=true、preserveConstEnums=true，正常项目仍保留上一节的内联配置。

以下片段来自独立的 type-errors.ts：

```typescript
declare const enum ExternalPhase { Ready = 1 }
const externalValue = ExternalPhase.Ready; // isolatedModules 禁止引用环境 const enum 成员。
```

## 6 枚举、字面量联合与 as const 对象

只需要约束有限输入、不需要运行时常量集合时，字面量联合无需生成额外对象。需要可枚举的常量名称与普通 JavaScript 互通时，可用 as const 对象，再从其值取得类型；这里先用显式 typeof 属性类型避免提前组合键查询操作符。

普通 enum 适合既需要运行时对象又希望以枚举成员表达类型的接口，但会产生生成代码，数值枚举还带反向键。as const 的只读信息仅用于检查，不冻结对象；这三种写法都不能替代外部输入校验。

```typescript
type PlainStatus = "ready" | "failed";
const StatusValues = { Ready: "ready", Failed: "failed" } as const;
type ObjectStatus = typeof StatusValues.Ready | typeof StatusValues.Failed;
function describePlain(value: PlainStatus): string { return value; }
const objectStatus: ObjectStatus = StatusValues.Ready;
console.log(describePlain(objectStatus), Object.keys(StatusValues).join(",")); // ready Ready,Failed
```

## 本章小结

普通枚举会创建对象，数值成员具有反向映射；成员类型与原始字符串并不等同。const enum 的内联取决于编译方式和选项，跨包环境声明存在额外限制。只约束值集合时也可选择联合或常量对象。

## 练习

1. 构造从 10 开始的三成员数值枚举，并添加一个重复值别名；核对前向值和反向键被覆盖的名称。
2. 将字符串枚举改写为常量对象及其值类型，比较原始字符串能否赋值，并检查运行时对象键。
3. 比较 build:11 与 build:11:preserve 的 main.js：默认产物应没有 Access 对象声明，对照产物应有；运行输出应完全相同。
4. 说明为什么只发布带 declare const enum 的声明、却不提供一致的构建与消费条件，会在 isolatedModules 或依赖版本变化时产生问题。

### 提示

1. 先写 First = 10，再写两个未初始化成员，最后写 Alias = First。
2. 用 typeof 常量对象.属性的联合提取值类型。
3. 分别检查两个输出目录，不能用一次生成物代表两套配置。
4. 对照独立反例的 isolatedModules 与声明文件不提供实现这两个条件。


### 参考解析

1. 前三项分别为 10、11、12；最后别名仍为 10，反向键 10 对应最后写入的 Alias。
2. as const 对象的值类型可接受普通字符串字面量 ready；原字符串枚举变量要求对应枚举成员。两种对象的键均可在运行时观察。
3. 默认配置内联 Access.All 为 3，preserve 对照保留 Access 声明；本例调用处仍内联，五行运行结果相同。
4. 环境 const enum 成员不能在 isolatedModules 下依靠跨文件替换；编译时内联的值也可能与运行依赖版本不一致，单有 .d.ts 不能解决这两项条件。


## 参考与引用来源

| 来源 | 支持的知识点与定位 |
| --- | --- |
| TypeScript 官方文档 | [Enums：数值、字符串、计算成员、反向映射与 const enum 限制](https://www.typescriptlang.org/docs/handbook/enums.html)；[5.0：所有枚举的成员联合类型](https://www.typescriptlang.org/docs/handbook/release-notes/typescript-5-0.html#all-enums-are-union-enums)；[isolatedModules：环境 const enum](https://www.typescriptlang.org/tsconfig/isolatedModules.html)；[preserveConstEnums](https://www.typescriptlang.org/tsconfig/preserveConstEnums.html)；[3.4：const assertions](https://www.typescriptlang.org/docs/handbook/release-notes/typescript-3-4.html#const-assertions)。 |
| npm 官方文档 | [npm run（v11）](https://docs.npmjs.com/cli/v11/commands/npm-run/)：从本技术目录运行已配置脚本，并解析本地工具。 |
